# Supervised Learning
(all results will be printed at the end)
## 2.1 Classical ML Models (Logistic Regression & Gradient Boosted Tree)
We decided to mostly use the last-measured value as input feature as this the closest to death and should be most informative. The 4 static values will be kept (for static weight this is the first).

In [1]:
# Simple Feature Preprocessing
# For each patient: last-measured value of each dynamic variable + static variables

import numpy as np
import pandas as pd
import pickle

df_train = pd.read_parquet("processed/set_a_processed.parquet")
df_val   = pd.read_parquet("processed/set_b_processed.parquet")
df_test  = pd.read_parquet("processed/set_c_processed.parquet")

DYNAMIC_VARS = sorted([
    'ALP', 'ALT', 'AST', 'Albumin', 'BUN', 'Bilirubin', 'Cholesterol',
    'Creatinine', 'DiasABP', 'FiO2', 'GCS', 'Glucose', 'HCO3', 'HCT',
    'HR', 'K', 'Lactate', 'MAP', 'MechVent', 'Mg', 'NIDiasABP', 'NIMAP',
    'NISysABP', 'Na', 'PaCO2', 'PaO2', 'Platelets', 'RespRate', 'SaO2',
    'SysABP', 'Temp', 'TroponinI', 'TroponinT', 'Urine', 'WBC', 'Weight', 'pH'
])
STATIC_VARS = ['Age', 'Gender', 'Height', 'StaticWeight']
print(len(DYNAMIC_VARS))
print(len(STATIC_VARS))
ALL_FEATURES = DYNAMIC_VARS + STATIC_VARS  # 41 features total

def extract_simple_features(df): #extract the last observation of each patient("indicated by recordID"

    last_row = df.sort_values(["RecordID", "hour"]).groupby("RecordID").last().reset_index()
    
    features = last_row[["RecordID"] + ALL_FEATURES].copy()
    labels   = last_row[["RecordID", "In-hospital_death"]].copy()
    
    return features, labels

X_train, y_train = extract_simple_features(df_train)
X_val,   y_val   = extract_simple_features(df_val)
X_test,  y_test  = extract_simple_features(df_test)
#check the prevalence rate to see check if we should include something to balance the data and what are good values for the

37
4


In [2]:
# Logistic Regression baseline - add balanced because of low prevalence
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (roc_auc_score, average_precision_score,
                             classification_report)

# Prepare arrays (drop RecordID as it has no predictive value)
X_tr = X_train[ALL_FEATURES]
X_v  = X_val[ALL_FEATURES]
X_te = X_test[ALL_FEATURES]
y_tr = y_train["In-hospital_death"]
y_v  = y_val["In-hospital_death"]
y_te = y_test["In-hospital_death"]

lr = LogisticRegression(max_iter=1000, class_weight="balanced", random_state=42)
lr.fit(X_tr, y_tr)

#directly compute probabilites as we want to compute AuROC and AuPRC - extract the second probability, the one for dying
lr_proba_val = lr.predict_proba(X_v)[:, 1]
print(f"Val  AUROC: {roc_auc_score(y_v, lr_proba_val):.4f}  |  AUPRC: {average_precision_score(y_v, lr_proba_val):.4f}")

Val  AUROC: 0.8482  |  AUPRC: 0.5183


In [3]:
# Gradient Boosted Tree baseline with random initialization
from sklearn.ensemble import GradientBoostingClassifier

gbt = GradientBoostingClassifier(
    n_estimators=200,
    max_depth=4,
    learning_rate=0.1,
    subsample=0.8,
    random_state=42,
)
gbt.fit(X_tr, y_tr)
gbt_proba_val = gbt.predict_proba(X_v)[:, 1]
print(f"Val  AUROC: {roc_auc_score(y_v, gbt_proba_val):.4f}  |  AUPRC: {average_precision_score(y_v, gbt_proba_val):.4f}")

Val  AUROC: 0.8474  |  AUPRC: 0.5086


### 2.1.2 Hyperparameter Tuning with GridSearchCV
We use 5-fold stratified cross-validation on the training set, optimizing for AUROC - optimizing for average_precision lead to worse results in both categories. We didn't include this in the GridSearch anymore to reduce necessary compute. Now we can make use of the validation set for the hyperparameter tuning with gridsearch.

In [4]:
from sklearn.model_selection import GridSearchCV, StratifiedKFold

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# Logistic Regression Grid Search
# Let choose if ElasticNet and regularization strength as well as if the model should reweight because of the imbalanced data
lr_param_grid = {
    "C": [0.001, 0.01, 0.1],
    "l1_ratio": [0, 0.25, 0.75, 1.0],
    "class_weight": ["balanced", None],
}

lr_gs = GridSearchCV(
    LogisticRegression(solver="saga", max_iter=5000, random_state=42),
    param_grid=lr_param_grid,
    cv=cv,
    scoring="roc_auc",
    n_jobs=-1,
    verbose=1
)
lr_gs.fit(X_tr, y_tr)

print(lr_gs.best_params_)
print(f"Best CV AUROC: {lr_gs.best_score_:.4f}")

Fitting 5 folds for each of 24 candidates, totalling 120 fits
{'C': 0.01, 'class_weight': 'balanced', 'l1_ratio': 0.25}
Best CV AUROC: 0.8293


In [5]:
# GBT Grid Search
gbt_param_grid = {
    "n_estimators": [50, 100],
    "max_depth": [5, 6],
    "learning_rate": [0.03, 0.05],
    "subsample": [0.6, 0.7],
    "min_samples_leaf": [15, 20],
}

gbt_gs = GridSearchCV(
    GradientBoostingClassifier(random_state=42),
    param_grid=gbt_param_grid,
    cv=cv,
    scoring="roc_auc",
    n_jobs=-1,
    verbose=1,
)
gbt_gs.fit(X_tr, y_tr)

print(gbt_gs.best_params_)
print(f"Best CV AUROC: {gbt_gs.best_score_:.4f}")

Fitting 5 folds for each of 32 candidates, totalling 160 fits
{'learning_rate': 0.05, 'max_depth': 5, 'min_samples_leaf': 20, 'n_estimators': 100, 'subsample': 0.7}
Best CV AUROC: 0.8552


In [6]:
# Evaluate tuned models on val & test
from sklearn.metrics import roc_auc_score, average_precision_score

print(f"{'Model':15s}  {'Val AUROC':>10s}  {'Val AUPRC':>10s}  {'Test AUROC':>11s}  {'Test AUPRC':>11s}")
for name, model in [("LR (tuned)", lr_gs.best_estimator_), ("GBT (tuned)", gbt_gs.best_estimator_)]:
    p_val  = model.predict_proba(X_v)[:, 1]
    p_test = model.predict_proba(X_te)[:, 1]
    print(f"{name:15s}  {roc_auc_score(y_v, p_val):10.4f}  {average_precision_score(y_v, p_val):10.4f}"
          f"  {roc_auc_score(y_te, p_test):11.4f}  {average_precision_score(y_te, p_test):11.4f}")


Model             Val AUROC   Val AUPRC   Test AUROC   Test AUPRC
LR (tuned)           0.8506      0.5276       0.8460       0.4986
GBT (tuned)          0.8502      0.5204       0.8495       0.5270


### 2.1.3 Feature Engineering

The simple last-value features only use the final hour of each variable, discarding 47 hours of temporal information. We engineer additional features to capture:

- **Last value**: the most recent measurement (already used above)
- **First value**: admission baseline
- **First–last delta**: trajectory over the full 48h stay
- **Min / Max**: worst values during the stay
- **Mean / Std**: average level and variability (instability signals)
- **Missingness count**: number of hours with an actual measurement (sicker patients get tested more often)

#### Selected Features from Signal Processing

- **Absolute sum of changes(volatility)**
- **Linear trend slope**
- **Kurtosis (detects spiky outlier events)**
- **Count above/below mean (time spent in abnormal territory)**

In [7]:
# Feature Engineering
# Use scaled+imputed data for all features exept std, which is computed on raw data for caputring true variability.

# Data before imputation & scaling) for std calculation
df_train_raw = pd.read_parquet("processed/set_a.parquet")
df_val_raw   = pd.read_parquet("processed/set_b.parquet")
df_test_raw  = pd.read_parquet("processed/set_c.parquet")

def extract_engineered_features(df_scaled, df_raw):

    grp = df_scaled.groupby("RecordID")
    grp_raw = df_raw.groupby("RecordID")
    
    last  = grp[DYNAMIC_VARS].last()
    first = grp[DYNAMIC_VARS].first()
    mean  = grp[DYNAMIC_VARS].mean()
    vmin  = grp[DYNAMIC_VARS].min()
    vmax  = grp[DYNAMIC_VARS].max()
    delta = last - first
    std   = grp_raw[DYNAMIC_VARS].std().fillna(0)
    
    # Some signal-processing features suggested in the handout- we used .apply(lambda x: ...) to call function on the grouping
    # 1. volatility
    abs_sum_chg = grp[DYNAMIC_VARS].apply(lambda x: x.diff().abs().sum())
    
    # 2. Linear trend slope
    def linear_slope(series):
        y = series.dropna().values
        if len(y) < 2:
            return 0.0
        x = np.arange(len(y))
        return np.polyfit(x, y, 1)[0]
    
    trend = grp[DYNAMIC_VARS].agg(linear_slope)
    
    # 3. Kurtosis
    kurt = grp[DYNAMIC_VARS].apply(lambda x: x.kurtosis()).fillna(0)
    
    # 4. Count above/below some close region around the mean  (time spent in abnormal territory)
    count_above = grp[DYNAMIC_VARS].apply(lambda x: (x > x.mean()+ x.std()).sum())
    count_below = grp[DYNAMIC_VARS].apply(lambda x: (x < x.mean()- x.std()).sum())
    
    # Add columns to dataset
    last        = last.rename(columns={c: f"{c}_last" for c in DYNAMIC_VARS})
    first       = first.rename(columns={c: f"{c}_first" for c in DYNAMIC_VARS})
    mean        = mean.rename(columns={c: f"{c}_mean" for c in DYNAMIC_VARS})
    std         = std.rename(columns={c: f"{c}_std" for c in DYNAMIC_VARS})
    vmin        = vmin.rename(columns={c: f"{c}_min" for c in DYNAMIC_VARS})
    vmax        = vmax.rename(columns={c: f"{c}_max" for c in DYNAMIC_VARS})
    delta       = delta.rename(columns={c: f"{c}_delta" for c in DYNAMIC_VARS})
    abs_sum_chg = abs_sum_chg.rename(columns={c: f"{c}_abs_sum_chg" for c in DYNAMIC_VARS})
    trend       = trend.rename(columns={c: f"{c}_trend" for c in DYNAMIC_VARS})
    kurt        = kurt.rename(columns={c: f"{c}_kurtosis" for c in DYNAMIC_VARS})
    count_above = count_above.rename(columns={c: f"{c}_count_above" for c in DYNAMIC_VARS})
    count_below = count_below.rename(columns={c: f"{c}_count_below" for c in DYNAMIC_VARS})
    
    statics = grp[STATIC_VARS].first()
    labels  = grp["In-hospital_death"].first().reset_index()
    
    features = pd.concat([last, first, mean, std, vmin, vmax, delta,
                           abs_sum_chg, trend, kurt, count_above, count_below,
                           statics], axis=1)
    features = features.reset_index()
    
    return features, labels

X_train_eng, y_train_eng = extract_engineered_features(df_train, df_train_raw)
X_val_eng,   y_val_eng   = extract_engineered_features(df_val,   df_val_raw)
X_test_eng,  y_test_eng  = extract_engineered_features(df_test,  df_test_raw)

ENG_FEATURES = [c for c in X_train_eng.columns if c != "RecordID"]

n_new = 5 * len(DYNAMIC_VARS)
print(f"Engineered feature count: {len(ENG_FEATURES)}")
X_train_eng.head()

Engineered feature count: 448


,RecordID,ALP_last,ALT_last,AST_last,Albumin_last,BUN_last,Bilirubin_last,Cholesterol_last,Creatinine_last,DiasABP_last,...,TroponinI_count_below,TroponinT_count_below,Urine_count_below,WBC_count_below,Weight_count_below,pH_count_below,Age,Gender,Height,StaticWeight
0,132539,0.666667,0.222222,-0.695652,-0.218305,-0.6875,-0.333333,-0.374775,-0.333333,0.279561,...,0,0,2,15,0,0,-0.583612,0.0,-0.451583,-2.437098
1,132540,-0.518519,-0.055556,0.217391,-0.446695,0.1250,0.666667,-0.361386,0.666667,-0.704886,...,0,0,0,13,20,12,0.669324,1.0,0.353422,0.139491
2,132541,1.074074,2.555556,5.217391,-1.588648,-1.0000,7.000000,1.687192,-1.000000,0.853822,...,0,0,3,0,0,16,-1.153129,0.0,-0.451583,-0.637229
3,132543,1.074074,-0.944444,-1.260870,3.207554,-0.5625,-1.666667,-0.160545,-0.333333,0.033449,...,0,0,20,0,0,0,0.213711,1.0,0.670353,0.233071
4,132545,0.037037,-0.166667,-0.130435,0.695258,0.3750,-0.666667,0.107243,0.166667,-0.294700,...,0,0,1,0,0,0,1.352744,0.0,-0.939658,-2.437098


In [8]:
# Extract engineered feature arrays and labels
X_tr_eng = X_train_eng[ENG_FEATURES].values
X_v_eng  = X_val_eng[ENG_FEATURES].values
X_te_eng = X_test_eng[ENG_FEATURES].values

y_tr_eng = y_train_eng["In-hospital_death"].values
y_v_eng  = y_val_eng["In-hospital_death"].values
y_te_eng = y_test_eng["In-hospital_death"].values

print(f"Feature matrices — Train: {X_tr_eng.shape}, Val: {X_v_eng.shape}, Test: {X_te_eng.shape}")


Feature matrices — Train: (4000, 448), Val: (4000, 448), Test: (4000, 448)


In [9]:
# Step 1: ElasticNet LR on all 448 features for feature selection
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline

lr_gs_eng = GridSearchCV(
    Pipeline([
        ("scaler", StandardScaler()),
        ("lr", LogisticRegression(penalty="elasticnet", solver="saga", max_iter=7000, random_state=42)),
    ]),
    param_grid={
        "lr__C": [0.001, 0.01, 0.1],
        "lr__l1_ratio": [0.5, 0.75, 1.0],
        "lr__class_weight": ["balanced", None],
    },
    cv=cv, scoring="roc_auc", n_jobs=-1, verbose=1,
)
lr_gs_eng.fit(X_tr_eng, y_tr_eng)
print(f"\nLR best params: {lr_gs_eng.best_params_}")
print(f"LR best CV AUROC: {lr_gs_eng.best_score_:.4f}")


Fitting 5 folds for each of 18 candidates, totalling 90 fits


/Users/ferdinandunterhuber/ETH/Machine-Learning-for-Health-Care/.venv/lib/python3.12/site-packages/sklearn/linear_model/_logistic.py:1135: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', and C=np.inf instead of penalty=None.
  warnings.warn(
/Users/ferdinandunterhuber/ETH/Machine-Learning-for-Health-Care/.venv/lib/python3.12/site-packages/sklearn/linear_model/_logistic.py:1135: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', and C=np.inf instead of penalty=None.
  warnings.warn(
/Users/ferdinandunterhuber/ETH/Machine-Learning-for-Health-Care/.venv/lib/python3.12/site-pack


LR best params: {'lr__C': 0.1, 'lr__class_weight': None, 'lr__l1_ratio': 1.0}
LR best CV AUROC: 0.8333


### Feature Selection via ElasticNet

The tuned ElasticNet LR evaluates all 448 features simultaneously and shrinks unimportant coefficients to zero.
We extract the non-zero features and retrain the GBT on only those — reducing dimensionality and speeding up the GBT grid search.


In [10]:
# ── Extract non-zero features from ElasticNet LR ─────────────────────────────
lr_coefs = lr_gs_eng.best_estimator_.named_steps["lr"].coef_[0]
nonzero_mask = lr_coefs != 0
selected_features = [f for f, m in zip(ENG_FEATURES, nonzero_mask) if m]

print(f"ElasticNet LR selected {len(selected_features)} / {len(ENG_FEATURES)} features")
print(f"Dropped {len(ENG_FEATURES) - len(selected_features)} features (coef = 0)\n")

# Show top features by absolute coefficient
coef_series = pd.Series(lr_coefs, index=ENG_FEATURES)
top_features = coef_series[nonzero_mask].abs().sort_values(ascending=False).head(20)
print("Top 20 features by |coefficient|:")
for feat, val in top_features.items():
    sign = "+" if coef_series[feat] > 0 else "-"
    print(f"  {sign} {feat:<30s}  |coef| = {val:.4f}")

# Build reduced feature arrays
sel_idx = [ENG_FEATURES.index(f) for f in selected_features]
X_tr_sel = X_tr_eng[:, sel_idx]
X_v_sel  = X_v_eng[:, sel_idx]
X_te_sel = X_te_eng[:, sel_idx]
print(f"\nReduced feature matrices — Train: {X_tr_sel.shape}, Val: {X_v_sel.shape}, Test: {X_te_sel.shape}")


ElasticNet LR selected 186 / 448 features
Dropped 262 features (coef = 0)

Top 20 features by |coefficient|:
  - GCS_last                        |coef| = 0.6043
  - MechVent_max                    |coef| = 0.3575
  - pH_min                          |coef| = 0.3396
  + pH_mean                         |coef| = 0.3213
  + Age                             |coef| = 0.2342
  - GCS_mean                        |coef| = 0.2172
  - Temp_mean                       |coef| = 0.2068
  + Platelets_count_above           |coef| = 0.1990
  - PaCO2_abs_sum_chg               |coef| = 0.1973
  + BUN_min                         |coef| = 0.1697
  - Cholesterol_count_above         |coef| = 0.1570
  - GCS_abs_sum_chg                 |coef| = 0.1539
  - RespRate_abs_sum_chg            |coef| = 0.1504
  + PaO2_mean                       |coef| = 0.1427
  + SysABP_first                    |coef| = 0.1418
  + Glucose_last                    |coef| = 0.1399
  - RespRate_trend                  |coef| = 0.1364
  - Dia

In [11]:
#  Step 2: GBT on ElasticNet-selected features
gbt_gs_eng = GridSearchCV(
    GradientBoostingClassifier(random_state=42),
    param_grid={
        "n_estimators": [100, 200],
        "max_depth": [4, 5],
        "learning_rate": [0.05, 0.1],
        "subsample": [0.7, 0.8],
        "min_samples_leaf": [10, 20],
    },
    cv=cv, scoring="roc_auc", n_jobs=-1, verbose=1,
)
gbt_gs_eng.fit(X_tr_sel, y_tr_eng)
print(f"\nGBT best params: {gbt_gs_eng.best_params_}")
print(f"GBT best CV AUROC: {gbt_gs_eng.best_score_:.4f}")


Fitting 5 folds for each of 32 candidates, totalling 160 fits

GBT best params: {'learning_rate': 0.05, 'max_depth': 4, 'min_samples_leaf': 20, 'n_estimators': 200, 'subsample': 0.8}
GBT best CV AUROC: 0.8592


In [12]:
#Save all CPU (sklearn) model results for use on cluster
import pickle, os
os.makedirs("models", exist_ok=True)

cpu_models = {
    # Simple features
    "lr_simple": lr_gs.best_estimator_,
    "gbt_simple": gbt_gs.best_estimator_,
    # Engineered features
    "lr_eng": lr_gs_eng.best_estimator_,
    "gbt_eng": gbt_gs_eng.best_estimator_,
    # Feature lists and data needed for prediction
    "selected_features": selected_features,
    "ENG_FEATURES": ENG_FEATURES,
}

with open("models/cpu_models.pkl", "wb") as f:
    pickle.dump(cpu_models, f)


## 2.2 RNN

In [24]:
#fist basline approaches
import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader
from sklearn.metrics import roc_auc_score, average_precision_score, roc_curve, precision_recall_curve
import matplotlib.pyplot as plt

#  1. Load parquet as tensors
def df_to_tensors(df, dynamic_vars, static_vars, n_steps=49):
    df = df.sort_values(["RecordID", "hour"])
    n_patients = df["RecordID"].nunique()
    X_dyn  = torch.tensor(df[dynamic_vars].values.reshape(n_patients, n_steps, -1), dtype=torch.float32)
    X_stat = torch.tensor(df.groupby("RecordID")[static_vars].first().values, dtype=torch.float32)
    y      = torch.tensor(df.groupby("RecordID")["In-hospital_death"].first().values, dtype=torch.float32)
    return X_dyn, X_stat, y

X_dyn_tr, X_stat_tr, y_tr_rnn = df_to_tensors(df_train, DYNAMIC_VARS, STATIC_VARS)
X_dyn_v,  X_stat_v,  y_v_rnn  = df_to_tensors(df_val,   DYNAMIC_VARS, STATIC_VARS)
X_dyn_te, X_stat_te, y_te_rnn = df_to_tensors(df_test,  DYNAMIC_VARS, STATIC_VARS)


# 2. Load data
BATCH = 128
train_dl = DataLoader(TensorDataset(X_dyn_tr, X_stat_tr, y_tr_rnn), batch_size=BATCH, shuffle=True)
val_dl   = DataLoader(TensorDataset(X_dyn_v,  X_stat_v,  y_v_rnn),  batch_size=BATCH)
test_dl  = DataLoader(TensorDataset(X_dyn_te, X_stat_te, y_te_rnn), batch_size=BATCH)

# 3. Using Torch LSTM - the static features are only concatenated at the end and given for the head as input
class LSTM(nn.Module):
    def __init__(self, input_dim=37, static_dim=4, hidden_dim=128, n_layers=1, dropout=0.3):
        super().__init__()
        self.lstm = nn.LSTM(
            input_size=input_dim,
            hidden_size=hidden_dim,
            num_layers=n_layers,
            batch_first=True,
            dropout=dropout if n_layers > 1 else 0.0,
        )
        self.head = nn.Linear(hidden_dim + static_dim, 1)

    def forward(self, x_dyn, x_stat):
        _, (h_n, _) = self.lstm(x_dyn)
        last_hidden = h_n[-1]                          # (batch, hidden)
        combined = torch.cat([last_hidden, x_stat], dim=1)
        return self.head(combined).squeeze(-1)

device = torch.device("cuda" if torch.cuda.is_available() else
                       "mps" if torch.backends.mps.is_available() else "cpu")
print(f"Device: {device}")

model = LSTM().to(device)
print(f"Parameters: {sum(p.numel() for p in model.parameters()):,}")

# 4. Training
pos_weight = torch.tensor([(y_tr_rnn == 0).sum() / (y_tr_rnn == 1).sum()]).to(device)
criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3, weight_decay=1e-3)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, patience=5, factor=0.5)

EPOCHS = 50
best_val_auc = 0
#add some history for plotting later
patience_ctr, patience_limit = 0, 10
history = {"train_loss": [], "val_auc": [], "val_prc": []}

for epoch in range(1, EPOCHS + 1):
    model.train()
    total_loss = 0
    for xd, xs, yb in train_dl:
        xd, xs, yb = xd.to(device), xs.to(device), yb.to(device)
        optimizer.zero_grad()
        loss = criterion(model(xd, xs), yb)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        total_loss += loss.item() * len(xd)
    avg_loss = total_loss / len(train_dl.dataset)

    model.eval()
    preds, labels = [], []
    with torch.no_grad():
        for xd, xs, yb in val_dl:
            preds.append(torch.sigmoid(model(xd.to(device), xs.to(device))).cpu())
            labels.append(yb)
    preds  = torch.cat(preds).numpy()
    labels = torch.cat(labels).numpy()
    val_auc = roc_auc_score(labels, preds)
    val_prc = average_precision_score(labels, preds)
    scheduler.step(val_auc)

    history["train_loss"].append(avg_loss)
    history["val_auc"].append(val_auc)
    history["val_prc"].append(val_prc)

    if val_auc > best_val_auc:
        best_val_auc = val_auc
        best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
        patience_ctr = 0
    else:
        patience_ctr += 1

    if epoch % 5 == 0 or patience_ctr == 0:
        print(f"Epoch {epoch:3d}  loss={avg_loss:.4f}  val_AUROC={val_auc:.4f}  val_AUPRC={val_prc:.4f}  lr={optimizer.param_groups[0]['lr']:.1e}")

    if patience_ctr >= patience_limit:
        print(f"Early stopping at epoch {epoch}")
        break

# 5. Test evaluation
model.load_state_dict(best_state)
model.eval()
preds_test, labels_test = [], []
with torch.no_grad():
    for xd, xs, yb in test_dl:
        preds_test.append(torch.sigmoid(model(xd.to(device), xs.to(device))).cpu())
        labels_test.append(yb)
preds_test  = torch.cat(preds_test).numpy()
labels_test = torch.cat(labels_test).numpy()

test_auc = roc_auc_score(labels_test, preds_test)
test_prc = average_precision_score(labels_test, preds_test)
print(f"\n{'LSTM (unidirectional)':<30s}  Val AUROC={best_val_auc:.4f}  Test AUROC={test_auc:.4f}  Test AUPRC={test_prc:.4f}")


Device: mps
Parameters: 85,637
Epoch   1  loss=1.0809  val_AUROC=0.8184  val_AUPRC=0.4427  lr=1.0e-03
Epoch   2  loss=0.8796  val_AUROC=0.8438  val_AUPRC=0.4848  lr=1.0e-03
Epoch   3  loss=0.8173  val_AUROC=0.8502  val_AUPRC=0.4962  lr=1.0e-03
Epoch   4  loss=0.7664  val_AUROC=0.8524  val_AUPRC=0.4925  lr=1.0e-03
Epoch   5  loss=0.7344  val_AUROC=0.8479  val_AUPRC=0.4976  lr=1.0e-03
Epoch  10  loss=0.5369  val_AUROC=0.8342  val_AUPRC=0.4610  lr=5.0e-04
Early stopping at epoch 14

LSTM (unidirectional)           Val AUROC=0.8524  Test AUROC=0.8396  Test AUPRC=0.4833


In [25]:
# Bidirectional LSTM - same archtecture but adding the branch back

class BiLSTM(nn.Module):
    def __init__(self, input_dim=37, static_dim=4, hidden_dim=128, n_layers=1, dropout=0.5):
        super().__init__()
        self.lstm = nn.LSTM(
            input_size=input_dim,
            hidden_size=hidden_dim,
            num_layers=n_layers,
            batch_first=True,
            dropout=dropout if n_layers > 1 else 0.0,
            bidirectional=True,
        )
        self.head = nn.Linear(2 * hidden_dim + static_dim, 1)

    def forward(self, x_dyn, x_stat):
        _, (h_n, _) = self.lstm(x_dyn)
        fwd = h_n[-2]   # last layer, forward
        bwd = h_n[-1]   # last layer, backward
        combined = torch.cat([fwd, bwd, x_stat], dim=1)
        return self.head(combined).squeeze(-1)

bi_model = BiLSTM().to(device)
print(bi_model)
print(f"Parameters: {sum(p.numel() for p in bi_model.parameters()):,}")

# Training
bi_criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
bi_optimizer = torch.optim.Adam(bi_model.parameters(), lr=5e-4, weight_decay=1e-3)
bi_scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(bi_optimizer, patience=5, factor=0.5)

best_bi_auc = 0
bi_patience, bi_patience_limit = 0, 10
bi_history = {"train_loss": [], "val_auc": [], "val_prc": []}

for epoch in range(1, EPOCHS + 1):
    bi_model.train()
    total_loss = 0
    for xd, xs, yb in train_dl:
        xd, xs, yb = xd.to(device), xs.to(device), yb.to(device)
        bi_optimizer.zero_grad()
        loss = bi_criterion(bi_model(xd, xs), yb)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(bi_model.parameters(), 1.0)
        bi_optimizer.step()
        total_loss += loss.item() * len(xd)
    avg_loss = total_loss / len(train_dl.dataset)

    bi_model.eval()
    preds, labels = [], []
    with torch.no_grad():
        for xd, xs, yb in val_dl:
            preds.append(torch.sigmoid(bi_model(xd.to(device), xs.to(device))).cpu())
            labels.append(yb)
    preds  = torch.cat(preds).numpy()
    labels = torch.cat(labels).numpy()
    val_auc = roc_auc_score(labels, preds)
    val_prc = average_precision_score(labels, preds)
    bi_scheduler.step(val_auc)

    bi_history["train_loss"].append(avg_loss)
    bi_history["val_auc"].append(val_auc)
    bi_history["val_prc"].append(val_prc)

    if val_auc > best_bi_auc:
        best_bi_auc = val_auc
        best_bi_state = {k: v.cpu().clone() for k, v in bi_model.state_dict().items()}
        bi_patience = 0
    else:
        bi_patience += 1

    if epoch % 5 == 0 or bi_patience == 0:
        print(f"Epoch {epoch:3d}  loss={avg_loss:.4f}  val_AUROC={val_auc:.4f}  val_AUPRC={val_prc:.4f}  lr={bi_optimizer.param_groups[0]['lr']:.1e}")

    if bi_patience >= bi_patience_limit:
        print(f"Early stopping at epoch {epoch}")
        break



BiLSTM(
  (lstm): LSTM(37, 128, batch_first=True, bidirectional=True)
  (head): Linear(in_features=260, out_features=1, bias=True)
)
Parameters: 171,269
Epoch   1  loss=1.1197  val_AUROC=0.7943  val_AUPRC=0.4031  lr=5.0e-04
Epoch   2  loss=0.9566  val_AUROC=0.8230  val_AUPRC=0.4395  lr=5.0e-04
Epoch   3  loss=0.8457  val_AUROC=0.8434  val_AUPRC=0.4779  lr=5.0e-04
Epoch   4  loss=0.7910  val_AUROC=0.8456  val_AUPRC=0.4859  lr=5.0e-04
Epoch   5  loss=0.7481  val_AUROC=0.8486  val_AUPRC=0.4770  lr=5.0e-04
Epoch  10  loss=0.5673  val_AUROC=0.8401  val_AUPRC=0.4721  lr=2.5e-04
Epoch  15  loss=0.4540  val_AUROC=0.8288  val_AUPRC=0.4549  lr=1.3e-04
Early stopping at epoch 15


## 2.3 a Transformer

In [26]:
# Transformer for Mortality Prediction - we use fixed Sinusoidal positional encodings for simplicity
import math

class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=50):
        super().__init__()
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(position * div_term)
        if d_model % 2 == 0:
            pe[:, 1::2] = torch.cos(position * div_term)
        else:
            pe[:, 1::2] = torch.cos(position * div_term[:-1])
        self.register_buffer("pe", pe.unsqueeze(0))

    def forward(self, x):
        return x + self.pe[:, :x.size(1)]

class Transformer(nn.Module):
    def __init__(self, input_dim=37, static_dim=4, d_model=64, nhead=4,
                 n_layers=2, dim_ff=128, dropout=0.3):
        super().__init__()
        # Project input features to d_model
        self.input_proj = nn.Linear(input_dim, d_model)
        self.pos_enc = PositionalEncoding(d_model, max_len=50)
        self.dropout = nn.Dropout(dropout)

        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model,
            nhead=nhead,
            dim_feedforward=dim_ff,
            dropout=dropout,
            batch_first=True,
        )
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=n_layers)

        # Learnable [CLS] token for classification
        self.cls_token = nn.Parameter(torch.randn(1, 1, d_model))
        self.head = nn.Linear(d_model + static_dim, 1)

    def forward(self, x_dyn, x_stat):
        B, T, _ = x_dyn.shape

        # Project dynamic features to d_model and add positional encoding
        x = self.dropout(self.pos_enc(self.input_proj(x_dyn)))

        # Prepend [CLS] token
        cls = self.cls_token.expand(B, -1, -1)
        x = torch.cat([cls, x], dim=1)

        # Transformer encoder
        x = self.transformer(x)                              # (B, 50, d_model)

        # Classify from [CLS] token
        cls_out = x[:, 0]  # (B, d_model)
        return self.head(torch.cat([cls_out, x_stat], dim=1)).squeeze(-1)

transformer = Transformer().to(device)

# Training
tx_criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
tx_optimizer = torch.optim.Adam(transformer.parameters(), lr=5e-4, weight_decay=1e-4)
tx_scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(tx_optimizer, patience=5, factor=0.5)

best_tx_auc = 0
tx_patience, tx_patience_limit = 0, 10
tx_history = {"train_loss": [], "val_auc": [], "val_prc": []}

for epoch in range(1, EPOCHS + 1):
    transformer.train()
    total_loss = 0
    for xd, xs, yb in train_dl:
        xd, xs, yb = xd.to(device), xs.to(device), yb.to(device)
        tx_optimizer.zero_grad()
        loss = tx_criterion(transformer(xd, xs), yb)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(transformer.parameters(), 1.0)
        tx_optimizer.step()
        total_loss += loss.item() * len(xd)
    avg_loss = total_loss / len(train_dl.dataset)

    transformer.eval()
    preds, labels = [], []
    with torch.no_grad():
        for xd, xs, yb in val_dl:
            preds.append(torch.sigmoid(transformer(xd.to(device), xs.to(device))).cpu())
            labels.append(yb)
    preds  = torch.cat(preds).numpy()
    labels = torch.cat(labels).numpy()
    val_auc = roc_auc_score(labels, preds)
    val_prc = average_precision_score(labels, preds)
    tx_scheduler.step(val_auc)

    tx_history["train_loss"].append(avg_loss)
    tx_history["val_auc"].append(val_auc)
    tx_history["val_prc"].append(val_prc)

    if val_auc > best_tx_auc:
        best_tx_auc = val_auc
        best_tx_state = {k: v.cpu().clone() for k, v in transformer.state_dict().items()}
        tx_patience = 0
    else:
        tx_patience += 1

    if epoch % 5 == 0 or tx_patience == 0:
        print(f"Epoch {epoch:3d}  loss={avg_loss:.4f}  val_AUROC={val_auc:.4f}  val_AUPRC={val_prc:.4f}  lr={tx_optimizer.param_groups[0]['lr']:.1e}")

    if tx_patience >= tx_patience_limit:
        print(f"Early stopping at epoch {epoch}")
        break

print(f"\nTransformer  Val AUROC={best_tx_auc:.4f} ")

Transformer(
  (input_proj): Linear(in_features=37, out_features=64, bias=True)
  (pos_enc): PositionalEncoding()
  (dropout): Dropout(p=0.3, inplace=False)
  (transformer): TransformerEncoder(
    (layers): ModuleList(
      (0-1): 2 x TransformerEncoderLayer(
        (self_attn): MultiheadAttention(
          (out_proj): NonDynamicallyQuantizableLinear(in_features=64, out_features=64, bias=True)
        )
        (linear1): Linear(in_features=64, out_features=128, bias=True)
        (dropout): Dropout(p=0.3, inplace=False)
        (linear2): Linear(in_features=128, out_features=64, bias=True)
        (norm1): LayerNorm((64,), eps=1e-05, elementwise_affine=True)
        (norm2): LayerNorm((64,), eps=1e-05, elementwise_affine=True)
        (dropout1): Dropout(p=0.3, inplace=False)
        (dropout2): Dropout(p=0.3, inplace=False)
      )
    )
  )
  (head): Linear(in_features=68, out_features=1, bias=True)
)
Parameters: 69,509
Epoch   1  loss=1.1273  val_AUROC=0.7658  val_AUPRC=0.3696 

In [27]:
# Hyperparameter Tuning for all DL models
from itertools import product

def train_and_eval(model, train_dl, val_dl, device, lr, wd, epochs=50, patience_limit=10):
    """Train a model, return best val AUROC, AUPRC, and state dict."""
    pos_w = torch.tensor([(y_tr_rnn == 0).sum() / (y_tr_rnn == 1).sum()]).to(device)
    criterion = nn.BCEWithLogitsLoss(pos_weight=pos_w)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=wd)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, patience=5, factor=0.5)

    best_auc, best_state, patience_ctr = 0, None, 0
    for epoch in range(1, epochs + 1):
        model.train()
        for xd, xs, yb in train_dl:
            xd, xs, yb = xd.to(device), xs.to(device), yb.to(device)
            optimizer.zero_grad()
            loss = criterion(model(xd, xs), yb)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()

        model.eval()
        preds, labels = [], []
        with torch.no_grad():
            for xd, xs, yb in val_dl:
                preds.append(torch.sigmoid(model(xd.to(device), xs.to(device))).cpu())
                labels.append(yb)
        preds  = torch.cat(preds).numpy()
        labels = torch.cat(labels).numpy()
        val_auc = roc_auc_score(labels, preds)
        scheduler.step(val_auc)

        if val_auc > best_auc:
            best_auc = val_auc
            best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
            patience_ctr = 0
        else:
            patience_ctr += 1
        if patience_ctr >= patience_limit:
            break

    # Eval best model on val for AUPRC too
    model.load_state_dict(best_state)
    model.eval()
    preds, labels = [], []
    with torch.no_grad():
        for xd, xs, yb in val_dl:
            preds.append(torch.sigmoid(model(xd.to(device), xs.to(device))).cpu())
            labels.append(yb)
    preds  = torch.cat(preds).numpy()
    labels = torch.cat(labels).numpy()
    best_prc = average_precision_score(labels, preds)
    return best_auc, best_prc, best_state

# Hyperparameter grids
lstm_grid = {
    "hidden_dim": [64, 128],
    "n_layers":   [1],
    "dropout":    [0.3, 0.5],
    "lr":         [1e-3, 5e-4],
    "wd":         [1e-3],
}

transformer_grid = {
    "d_model":  [64],
    "nhead":    [4],
    "n_layers": [2],
    "dim_ff":   [128, 256],
    "dropout":  [0.2, 0.3],
    "lr":       [5e-4, 1e-3],
    "wd":       [1e-4, 1e-3],
}

# Tune
print("Tuning LSTM ")
best_lstm_result = {"auc": 0}
lstm_keys = list(lstm_grid.keys())
for vals in product(*lstm_grid.values()):
    cfg = dict(zip(lstm_keys, vals))
    m = LSTM(
        hidden_dim=cfg["hidden_dim"], n_layers=cfg["n_layers"], dropout=cfg["dropout"]
    ).to(device)
    auc, prc, state = train_and_eval(m, train_dl, val_dl, device, cfg["lr"], cfg["wd"])
    if auc > best_lstm_result["auc"]:
        best_lstm_result = {"auc": auc, "prc": prc, "state": state, "cfg": cfg}

print(f" Best LSTM: AUROC={best_lstm_result['auc']:.4f} AUPRC={best_lstm_result['prc']:.4f}")
print(f"  Config: {best_lstm_result['cfg']}")

print("Tuning BiLSTM")
best_bilstm_result = {"auc": 0}
for vals in product(*lstm_grid.values()):
    cfg = dict(zip(lstm_keys, vals))
    m = BiLSTM(
        hidden_dim=cfg["hidden_dim"], n_layers=cfg["n_layers"], dropout=cfg["dropout"]
    ).to(device)
    auc, prc, state = train_and_eval(m, train_dl, val_dl, device, cfg["lr"], cfg["wd"])
    if auc > best_bilstm_result["auc"]:
        best_bilstm_result = {"auc": auc, "prc": prc, "state": state, "cfg": cfg}

print(f" Best BiLSTM: AUROC={best_bilstm_result['auc']:.4f} AUPRC={best_bilstm_result['prc']:.4f}")
print(f"  Config: {best_bilstm_result['cfg']}")

print("Tuning Transformer")
best_tx_result = {"auc": 0}
tx_keys = list(transformer_grid.keys())
for vals in product(*transformer_grid.values()):
    cfg = dict(zip(tx_keys, vals))
    # nhead must divide d_model
    if cfg["d_model"] % cfg["nhead"] != 0:
        continue
    m = Transformer(
        d_model=cfg["d_model"], nhead=cfg["nhead"], n_layers=cfg["n_layers"],
        dim_ff=cfg["dim_ff"], dropout=cfg["dropout"]
    ).to(device)
    auc, prc, state = train_and_eval(m, train_dl, val_dl, device, cfg["lr"], cfg["wd"])
    if auc > best_tx_result["auc"]:
        best_tx_result = {"auc": auc, "prc": prc, "state": state, "cfg": cfg}

print(f" Best Transformer: AUROC={best_tx_result['auc']:.4f} AUPRC={best_tx_result['prc']:.4f}")
print(f"  Config: {best_tx_result['cfg']}")

Tuning LSTM 


KeyboardInterrupt: 

## 2.3 b Set-based Representation (Horn et al.)

Instead of modeling a sequence of time steps (with imputed missing values), we model a
**sequence of actual measurements**. Each observation is encoded as a triplet **(t, z, v)**:
- **t** — scaled time in [0, 1]
- **z** — one-hot encoding of the variable (41 categories: 37 dynamic + 4 static)
- **v** — the scaled measurement value

This avoids imputation entirely and naturally handles irregular sampling. We feed the
triplet sequence into a Transformer.

In [29]:
# ── Triplet (t, z, v) encoding — Horn et al. ─────────────────────────────────
import pickle
from torch.nn.utils.rnn import pad_sequence
import warnings
warnings.filterwarnings("ignore", message="X does not have valid feature names")

# Load the scalers fitted in Q1.3
with open("processed/scalers.pkl", "rb") as f:
    scalers_dict = pickle.load(f)

# Build per-variable scaling: map each variable name to (scaler, col_index)
robust_scaler = scalers_dict["robust"]
standard_scaler = scalers_dict["standard"]
robust_cols = scalers_dict["robust_cols"]
standard_cols = scalers_dict["standard_cols"]

def scale_value(var, val):
    """Scale a single value using the correct scaler for that variable."""
    if var in robust_cols:
        idx = robust_cols.index(var)
        # RobustScaler expects 2D input
        dummy = np.zeros((1, len(robust_cols)))
        dummy[0, idx] = val
        return float(robust_scaler.transform(dummy)[0, idx])
    elif var in standard_cols:
        idx = standard_cols.index(var)
        dummy = np.zeros((1, len(standard_cols)))
        dummy[0, idx] = val
        return float(standard_scaler.transform(dummy)[0, idx])
    elif var == "MechVent":
        return val - 0.5  # binary shift
    else:
        return val  # Gender: unscaled

ALL_VARS = DYNAMIC_VARS + STATIC_VARS  # 41 variables
var_to_idx = {v: i for i, v in enumerate(ALL_VARS)}
N_VARS = len(ALL_VARS)

def build_triplets(df_raw):
    t_max = df_raw["hour"].max()
    triplets, labels = [], []

    for rid, grp in df_raw.groupby("RecordID"):
        obs = []
        labels.append(grp["In-hospital_death"].iloc[0])

        # Dynamic: one triplet per actual measurement (skip NaN)
        for _, row in grp.iterrows():
            t = row["hour"] / t_max
            for var in DYNAMIC_VARS:
                if pd.notna(row[var]):
                    v = scale_value(var, float(row[var]))
                    obs.append([t, v, var_to_idx[var]])

        # Static: add once at t=0
        row0 = grp.iloc[0]
        for var in STATIC_VARS:
            if pd.notna(row0[var]):
                v = scale_value(var, float(row0[var]))
                obs.append([0.0, v, var_to_idx[var]])

        triplets.append(np.array(obs, dtype=np.float32) if obs else np.zeros((1, 3), dtype=np.float32))

    return triplets, np.array(labels, dtype=np.float32)

df_train_raw = pd.read_parquet("processed/set_a.parquet")
df_val_raw   = pd.read_parquet("processed/set_b.parquet")
df_test_raw  = pd.read_parquet("processed/set_c.parquet")

trip_tr, y_trip_tr = build_triplets(df_train_raw)
trip_v,  y_trip_v  = build_triplets(df_val_raw)
trip_te, y_trip_te = build_triplets(df_test_raw)

lengths = [len(t) for t in trip_tr]
print(f"Patients: {len(trip_tr)}, median seq len: {np.median(lengths):.0f}, max: {max(lengths)}")


Patients: 4000
Triplets per patient — min: 5, median: 368, max: 652
Triplet format: [time, value, var_idx] — var_idx will be embedded in the model


In [ ]:
from torch.utils.data import TensorDataset, DataLoader
import torch
# Transformer on triplet sequences (learned variable embedding)

def collate_triplets(batch):
    seqs, labels = zip(*batch)
    lengths = [len(s) for s in seqs]
    seqs_padded = pad_sequence(seqs, batch_first=True, padding_value=0.0)
    max_len = seqs_padded.size(1)
    mask = torch.arange(max_len).unsqueeze(0) >= torch.tensor(lengths).unsqueeze(1)
    return seqs_padded, mask, torch.tensor(labels, dtype=torch.float32)

class TripletDataset(torch.utils.data.Dataset):
    def __init__(self, triplets, labels):
        self.triplets = [torch.tensor(t, dtype=torch.float32) for t in triplets]
        self.labels = labels
    def __len__(self):
        return len(self.triplets)
    def __getitem__(self, idx):
        return self.triplets[idx], self.labels[idx]

trip_train_dl = DataLoader(TripletDataset(trip_tr, y_trip_tr), batch_size=64, shuffle=True,  collate_fn=collate_triplets)
trip_val_dl   = DataLoader(TripletDataset(trip_v,  y_trip_v),  batch_size=64, shuffle=False, collate_fn=collate_triplets)
trip_test_dl  = DataLoader(TripletDataset(trip_te, y_trip_te), batch_size=64, shuffle=False, collate_fn=collate_triplets)

import os
os.environ['PYTORCH_ENABLE_MPS_FALLBACK'] = '1'

class TripletTransformer(nn.Module):
    def __init__(self, n_vars=N_VARS, embed_dim=16, d_model=128, nhead=4,
                 n_layers=3, dim_ff=256, dropout=0.2):
        super().__init__()
        # Learned embedding for variable identity (replaces 41-dim one-hot)
        self.var_embedding = nn.Embedding(n_vars, embed_dim)
        # Input = time (1) + value (1) + var_embedding (8) = 10
        self.input_proj = nn.Linear(2 + embed_dim, d_model)
        self.cls_token = nn.Parameter(torch.randn(1, 1, d_model))
        self.dropout = nn.Dropout(dropout)

        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model, nhead=nhead, dim_feedforward=dim_ff,
            dropout=dropout, batch_first=True,
        )
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=n_layers, enable_nested_tensor=False)
        self.head = nn.Linear(d_model, 1)

    def forward(self, x_padded, pad_mask):
        # x_padded: (B, L, 3) — columns: [time, value, var_idx]
        B, L, _ = x_padded.shape
        t_v = x_padded[:, :, :2]                                    # (B, L, 2) — time + value
        var_idx = x_padded[:, :, 2].long()                          # (B, L) — variable indices
        var_emb = self.var_embedding(var_idx)                        # (B, L, embed_dim)
        x = torch.cat([t_v, var_emb], dim=2)                        # (B, L, 2 + embed_dim)
        x = self.dropout(self.input_proj(x))                         # (B, L, d_model)

        cls = self.cls_token.expand(B, -1, -1)                      # (B, 1, d_model)
        x = torch.cat([cls, x], dim=1)                               # (B, L+1, d_model)
        cls_mask = torch.zeros(B, 1, dtype=torch.bool, device=pad_mask.device)
        mask = torch.cat([cls_mask, pad_mask], dim=1)
        x = self.transformer(x, src_key_padding_mask=mask)
        return self.head(x[:, 0]).squeeze(-1)

trip_tx = TripletTransformer().to(device)
print(trip_tx)
print(f"Parameters: {sum(p.numel() for p in trip_tx.parameters()):,}")

# Training
trip_pos_w = torch.tensor([(y_trip_tr == 0).sum() / (y_trip_tr == 1).sum()], dtype=torch.float32).to(device)
trip_crit = nn.BCEWithLogitsLoss(pos_weight=trip_pos_w)
trip_opt = torch.optim.Adam(trip_tx.parameters(), lr=1e-3, weight_decay=1e-4)
trip_sched = torch.optim.lr_scheduler.ReduceLROnPlateau(trip_opt, patience=7, factor=0.5)

best_trip_auc = 0
trip_pat = 0
trip_history = {"train_loss": [], "val_auc": [], "val_prc": []}

for epoch in range(1, EPOCHS + 1):
    trip_tx.train()
    total_loss = 0
    for x_pad, mask, yb in trip_train_dl:
        x_pad, mask, yb = x_pad.to(device), mask.to(device), yb.to(device)
        trip_opt.zero_grad()
        loss = trip_crit(trip_tx(x_pad, mask), yb)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(trip_tx.parameters(), 1.0)
        trip_opt.step()
        total_loss += loss.item() * len(yb)
    avg_loss = total_loss / len(trip_train_dl.dataset)

    trip_tx.eval()
    preds, labels = [], []
    with torch.no_grad():
        for x_pad, mask, yb in trip_val_dl:
            preds.append(torch.sigmoid(trip_tx(x_pad.to(device), mask.to(device))).cpu())
            labels.append(yb)
    preds  = torch.cat(preds).numpy()
    labels = torch.cat(labels).numpy()
    val_auc = roc_auc_score(labels, preds)
    val_prc = average_precision_score(labels, preds)
    trip_sched.step(val_auc)

    trip_history["train_loss"].append(avg_loss)
    trip_history["val_auc"].append(val_auc)
    trip_history["val_prc"].append(val_prc)

    if val_auc > best_trip_auc:
        best_trip_auc = val_auc
        best_trip_state = {k: v.cpu().clone() for k, v in trip_tx.state_dict().items()}
        trip_pat = 0
    else:
        trip_pat += 1
    if epoch % 5 == 0 or trip_pat == 0:
        print(f"Epoch {epoch:3d}  loss={avg_loss:.4f}  val_AUROC={val_auc:.4f}  val_AUPRC={val_prc:.4f}  lr={trip_opt.param_groups[0]['lr']:.1e}")
    if trip_pat >= 10:
        print(f"Early stopping at epoch {epoch}")
        break


TripletTransformer(
  (var_embedding): Embedding(41, 8)
  (input_proj): Linear(in_features=10, out_features=64, bias=True)
  (dropout): Dropout(p=0.3, inplace=False)
  (transformer): TransformerEncoder(
    (layers): ModuleList(
      (0-1): 2 x TransformerEncoderLayer(
        (self_attn): MultiheadAttention(
          (out_proj): NonDynamicallyQuantizableLinear(in_features=64, out_features=64, bias=True)
        )
        (linear1): Linear(in_features=64, out_features=128, bias=True)
        (dropout): Dropout(p=0.3, inplace=False)
        (linear2): Linear(in_features=128, out_features=64, bias=True)
        (norm1): LayerNorm((64,), eps=1e-05, elementwise_affine=True)
        (norm2): LayerNorm((64,), eps=1e-05, elementwise_affine=True)
        (dropout1): Dropout(p=0.3, inplace=False)
        (dropout2): Dropout(p=0.3, inplace=False)
      )
    )
  )
  (head): Linear(in_features=64, out_features=1, bias=True)
)
Parameters: 68,105


In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# Final Model Comparison — All model classes on the TEST set
# ══════════════════════════════════════════════════════════════════════════════
import pickle
import warnings
warnings.filterwarnings("ignore", message="X does not have valid feature names")
from sklearn.metrics import roc_auc_score, average_precision_score, roc_curve, precision_recall_curve

# Load CPU models from pickle (needed when running on cluster without re-running sklearn cells)
try:
    lr_gs  # check if already in memory
except NameError:
    with open("models/cpu_models.pkl", "rb") as f:
        cpu_models = pickle.load(f)
    class _Holder: pass
    lr_gs = _Holder(); lr_gs.best_estimator_ = cpu_models["lr_simple"]
    gbt_gs = _Holder(); gbt_gs.best_estimator_ = cpu_models["gbt_simple"]
    lr_gs_eng = _Holder(); lr_gs_eng.best_estimator_ = cpu_models["lr_eng"]
    gbt_gs_eng = _Holder(); gbt_gs_eng.best_estimator_ = cpu_models["gbt_eng"]
    selected_features = cpu_models["selected_features"]
    ENG_FEATURES = cpu_models["ENG_FEATURES"]
    print("Loaded CPU models from models/cpu_models.pkl")

# ── 1. Classical ML (simple features) ────────────────────────────────────────
print("=" * 70)
print("1. Classical ML — Simple Features (41)")
print("=" * 70)
classical_simple = []
for name, model in [("LR", lr_gs.best_estimator_), ("GBT", gbt_gs.best_estimator_)]:
    p = model.predict_proba(X_te)[:, 1]
    auc = roc_auc_score(y_te, p)
    prc = average_precision_score(y_te, p)
    classical_simple.append((name, auc, prc, p))
    print(f"  {name:<10s}  AUROC={auc:.4f}  AUPRC={prc:.4f}")

# ── 2. Classical ML (engineered features, ElasticNet-selected) ────────────────
print(f"\n{'=' * 70}")
print("2. Classical ML — Engineered Features (ElasticNet-selected)")
print("=" * 70)
classical_eng = []
# LR via pipeline (includes scaler)
p_lr_eng = lr_gs_eng.best_estimator_.predict_proba(X_te_eng)[:, 1]
auc_lr = roc_auc_score(y_te_eng, p_lr_eng)
prc_lr = average_precision_score(y_te_eng, p_lr_eng)
classical_eng.append(("LR (eng)", auc_lr, prc_lr, p_lr_eng))
print(f"  {'LR (eng)':<10s}  AUROC={auc_lr:.4f}  AUPRC={prc_lr:.4f}")
# GBT on selected features
p_gbt_eng = gbt_gs_eng.best_estimator_.predict_proba(X_te_sel)[:, 1]
auc_gbt = roc_auc_score(y_te_eng, p_gbt_eng)
prc_gbt = average_precision_score(y_te_eng, p_gbt_eng)
classical_eng.append(("GBT (eng)", auc_gbt, prc_gbt, p_gbt_eng))
print(f"  {'GBT (eng)':<10s}  AUROC={auc_gbt:.4f}  AUPRC={prc_gbt:.4f}")

#3. RNNs (tuned)
print(f"\n{'=' * 70}")
print("3. RNNs — Tuned")
print("=" * 70)
rnn_results = []
for name, ModelClass, result, is_tx in [
    ("LSTM (tuned)",   LSTM,       best_lstm_result,   False),
    ("BiLSTM (tuned)", BiLSTM,     best_bilstm_result, False),
]:
    cfg = result["cfg"]
    m = ModelClass(hidden_dim=cfg["hidden_dim"], n_layers=cfg["n_layers"],
                   dropout=cfg["dropout"]).to(device)
    m.load_state_dict(result["state"])
    m.eval()
    preds, labels = [], []
    with torch.no_grad():
        for xd, xs, yb in test_dl:
            preds.append(torch.sigmoid(m(xd.to(device), xs.to(device))).cpu())
            labels.append(yb)
    pt = torch.cat(preds).numpy()
    yt = torch.cat(labels).numpy()
    auc = roc_auc_score(yt, pt)
    prc = average_precision_score(yt, pt)
    rnn_results.append((name, auc, prc, pt, yt))
    print(f"  {name:<20s}  AUROC={auc:.4f}  AUPRC={prc:.4f}")

# ── 4. Transformers (tuned) ──────────────────────────────────────────────────
print(f"\n{'=' * 70}")
print("4. Transformers — Tuned")
print("=" * 70)
tx_results = []
# Tabular Transformer
cfg = best_tx_result["cfg"]
m = Transformer(d_model=cfg["d_model"], nhead=cfg["nhead"],
                n_layers=cfg["n_layers"], dim_ff=cfg["dim_ff"],
                dropout=cfg["dropout"]).to(device)
m.load_state_dict(best_tx_result["state"])
m.eval()
preds, labels = [], []
with torch.no_grad():
    for xd, xs, yb in test_dl:
        preds.append(torch.sigmoid(m(xd.to(device), xs.to(device))).cpu())
        labels.append(yb)
pt_tx = torch.cat(preds).numpy()
yt_tx = torch.cat(labels).numpy()
auc_tx = roc_auc_score(yt_tx, pt_tx)
prc_tx = average_precision_score(yt_tx, pt_tx)
tx_results.append(("Transformer (tuned)", auc_tx, prc_tx, pt_tx, yt_tx))
print(f"  {'Transformer (tuned)':<25s}  AUROC={auc_tx:.4f}  AUPRC={prc_tx:.4f}")
# Triplet Transformer
tx_results.append(("Triplet Transformer", trip_test_auc, trip_test_prc, trip_preds_test, trip_labels_test))
print(f"  {'Triplet Transformer':<25s}  AUROC={trip_test_auc:.4f}  AUPRC={trip_test_prc:.4f}")

# ══════════════════════════════════════════════════════════════════════════════
# 5. Best of each model class — head-to-head
# ══════════════════════════════════════════════════════════════════════════════
print(f"\n{'=' * 70}")
print("5. BEST OF EACH CLASS — Head-to-Head")
print("=" * 70)

# Pick best from each class by AUROC
best_classical = max(classical_simple + classical_eng, key=lambda x: x[1])
best_rnn = max(rnn_results, key=lambda x: x[1])
best_tx = max(tx_results, key=lambda x: x[1])

final = [
    (best_classical[0], best_classical[1], best_classical[2], best_classical[3], y_te),
    (best_rnn[0], best_rnn[1], best_rnn[2], best_rnn[3], best_rnn[4]),
    (best_tx[0], best_tx[1], best_tx[2], best_tx[3], best_tx[4]),
]

print(f"\n{'Model':<25s}  {'Test AUROC':>11s}  {'Test AUPRC':>11s}")
print("-" * 52)
for name, auc, prc, _, _ in final:
    print(f"{name:<25s}  {auc:11.4f}  {prc:11.4f}")

# ── ROC & PRC plots ──────────────────────────────────────────────────────────
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6))
for name, auc, prc, pt, yt in final:
    fpr, tpr, _ = roc_curve(yt, pt)
    ax1.plot(fpr, tpr, linewidth=2, label=f"{name} ({auc:.3f})")
    prec, rec, _ = precision_recall_curve(yt, pt)
    ax2.plot(rec, prec, linewidth=2, label=f"{name} ({prc:.3f})")

ax1.plot([0, 1], [0, 1], "k--", alpha=0.3)
ax1.set_xlabel("FPR"); ax1.set_ylabel("TPR")
ax1.set_title("ROC — Best of Each Model Class"); ax1.legend()
baseline = y_te.mean() if hasattr(y_te, 'mean') else np.mean(y_te)
ax2.axhline(baseline, color="k", linestyle="--", alpha=0.4, label=f"Baseline ({baseline:.3f})")
ax2.set_xlabel("Recall"); ax2.set_ylabel("Precision")
ax2.set_title("PRC — Best of Each Model Class"); ax2.legend()
plt.tight_layout(); plt.show()
